Goal:

- Model **claim severity** (how large claims are when they occur)
- Derive a **Technical Premium** to support pricing decisions

In [1]:
import pandas as pd

df = pd.read_csv("insurance_dataset.csv")

df.head()

,age,driver_type,years_experience,income_band,vehicle_age,vehicle_type,vehicle_value,airbags,tracking_device,region,...,annual_mileage,speeding_score,previous_claims,fraud_flag,risk_score,base_premium,premium,claim_occurred,claim_amount,claim_type
0,59,private,41,middle,0,SUV,3.005088e+06,2,0,Mombasa,...,15844,low,0,0,0.33,150254.402493,199838.355316,0,0.0,none
1,49,taxi,31,middle,10,sedan,5.223960e+05,6,0,Rural,...,35500,medium,1,0,0.04,26119.802262,27164.594353,0,0.0,none
2,35,private,17,low,8,sedan,8.075370e+05,0,0,Nairobi,...,13719,medium,1,0,0.50,40376.852016,60565.278024,0,0.0,none
3,63,taxi,45,low,6,sedan,8.082229e+05,4,1,Nairobi,...,31607,low,1,0,0.56,40411.145448,63041.386899,0,0.0,none
4,28,taxi,10,middle,0,sedan,8.101454e+05,1,1,Nairobi,...,26988,high,0,0,0.34,40507.271620,54279.743971,0,0.0,none


- The severity target column is **claim_amount**

### Gamma Severity Model

Gamma regression assumes claim amounts follow a Gamma distribution.

- Claim amounts are strictly positive - GD models positive values only
- Distribution is highly skewed (many small claims, few large ones) - GD models right-skewed and heavy tailed (large claims) data
- Variance increases with mean (heteroscedasticity) - bigger expected claims = more uncertainty
- Linear models with Gaussian assumptions perform poorly

#### Preparing severity dataset

In [3]:
from sklearn.model_selection import train_test_split

# We model severity using gamma only where claims exist
df_severity = df[df["claim_occurred"] == 1].copy()

# Gamma regression requires positive target values (claim_amount is already positive - good)
y_sev = df_severity["claim_amount"]

drop_cols = ["claim_occurred", "claim_amount", "premium", "base_premium", "risk_score", "claim_type"]
X_sev = df_severity.drop(columns=drop_cols)

In [4]:
X_sev["safety_score"] = (X_sev["airbags"] * 0.5 + X_sev["tracking_device"] * 2)

speed_map = {"low": 0, "medium": 1, "high": 2}
X_sev["behavior_score"] = (X_sev["annual_mileage"] / 10000 + X_sev["previous_claims"] * 2 + X_sev["speeding_score"].map(speed_map))

X_sev["age_band"] = pd.cut(X_sev["age"], bins=[20,30,50,70], labels=["young","mid","older"])
X_sev["vehicle_age_band"] = pd.cut(X_sev["vehicle_age"], bins=[0,3,7,15], labels=["new","mid","old"])
X_sev["high_mileage"] = (X_sev["annual_mileage"] > 25000).astype(int)

X_sev = pd.get_dummies(X_sev, drop_first=True)

In [5]:
X_train_sev, X_test_sev, y_train_sev, y_test_sev = train_test_split(X_sev, y_sev, test_size=0.2, random_state=42)

In [6]:
from sklearn.linear_model import GammaRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

gamma_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", GammaRegressor(alpha=0.1))
])

gamma_model.fit(X_train_sev, y_train_sev)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', GammaRegressor(alpha=0.1))])

GammaRegressor assumes:
- mean of claim amount is modeled as a function of features
- alpha controls regularization (prevents overfitting)

In [7]:
sev_pred = gamma_model.predict(X_test_sev)

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test_sev, sev_pred)
rmse = np.sqrt(mean_squared_error(y_test_sev, sev_pred))

print("Gamma Severity Model Performance")
print("MAE:", mae)
print("RMSE:", rmse)

Gamma Severity Model Performance
MAE: 237247.62972036915
RMSE: 355597.84914366785


- MAE = 237k - On average, model is off by ~238,000 KES per claim
- RMSE = 356k - large errors (big claims not well captured)

In [9]:
df_severity["claim_amount"].mean()

np.float64(562079.6809826754)

In [10]:
import pandas as pd

pd.DataFrame({"actual": y_test_sev, "pred": sev_pred}).corr()

,actual,pred
actual,1.000000,0.753512
pred,0.753512,1.000000


- High correlation - good severity model

In [11]:
df_eval = pd.DataFrame({"actual": y_test_sev, "pred": sev_pred})

df_eval["band"] = pd.qcut(df_eval["pred"], 5)

df_eval.groupby("band", observed=True)["actual"].mean()

band
(189792.04, 267111.69]       2.073545e+05
(267111.69, 383923.318]      3.233383e+05
(383923.318, 483912.614]     4.259653e+05
(483912.614, 792789.697]     6.328224e+05
(792789.697, 5630512.535]    1.237892e+06
Name: actual, dtype: float64

In [12]:
import joblib

model = joblib.load("xgb_model.pkl")

In [47]:
# Frequency model (from XGBoost)
freq = model.predict_proba(X_test_sev)[:, 1]

# Severity model (Gamma regression)
sev = gamma_model.predict(X_test_sev)

# Aligning indices
sev = pd.Series(sev, index=X_test_sev.index)

expected_loss = freq * sev

expense_loading = 0.3
profit_margin = 0.1

technical_premium = expected_loss * (1 + expense_loading + profit_margin)

In [48]:
results = X_test_sev.copy()

results["frequency"] = freq
results["severity"] = sev
results["expected_loss"] = expected_loss
results["technical_premium"] = technical_premium

In [49]:
results[["frequency", "severity", "expected_loss", "technical_premium"]].head()

,frequency,severity,expected_loss,technical_premium
20656,0.617921,4.800345e+05,296623.351764,415272.692470
38458,0.539152,4.083665e+05,220171.458268,308240.041575
42401,0.559773,7.119635e+05,398538.100060,557953.340084
1335,0.500418,1.200362e+06,600682.335484,840955.269678
22809,0.475293,4.358521e+05,207157.580665,290020.612931


In [50]:
results.sort_values("technical_premium", ascending=False).head()

,age,years_experience,vehicle_age,vehicle_value,airbags,tracking_device,policy_duration,annual_mileage,previous_claims,fraud_flag,...,speeding_score_low,speeding_score_medium,age_band_mid,age_band_older,vehicle_age_band_mid,vehicle_age_band_old,frequency,severity,expected_loss,technical_premium
37121,54,36,0,6.786959e+06,4,1,12,24508,1,0,...,True,False,False,True,False,False,0.601947,5.630513e+06,3.389269e+06,4.744977e+06
10746,64,46,0,5.471316e+06,1,1,12,38144,0,0,...,True,False,False,True,False,False,0.647751,3.435409e+06,2.225290e+06,3.115406e+06
32679,24,6,3,5.768044e+06,6,0,3,32925,0,0,...,False,False,False,False,False,False,0.634935,3.434842e+06,2.180902e+06,3.053263e+06
43544,36,18,0,5.183629e+06,0,0,3,28255,0,0,...,False,True,True,False,False,False,0.650467,3.134653e+06,2.038988e+06,2.854584e+06
16113,68,50,0,4.734562e+06,6,0,12,31096,1,0,...,True,False,False,True,False,False,0.737003,2.691415e+06,1.983580e+06,2.777011e+06


### Tweedie Regression Model

Tweedie models both frequency (claim yes/no) and severity (claim amount) at once (pure premium).

The Tweedie model is a compound distribution model used when:

- Many policies have zero claims
- Some have positive claim amounts
- Claim sizes are highly skewed

> Most people don’t claim, but when they do, amounts vary a lot

In [35]:
df = pd.read_csv("insurance_dataset.csv")

# Pure premium = actual claim amount (includes zeros)
y = df["claim_amount"]

- 0 for no claim
- positive value for claims

In [36]:
drop_cols = ["claim_amount", "claim_type", "claim_occurred", "premium", "base_premium", "risk_score"]

X = df.drop(columns=drop_cols)

In [37]:
X = pd.get_dummies(X, drop_first=True)

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [39]:
from sklearn.linear_model import TweedieRegressor

pipeline = Pipeline([
    ("scaler", StandardScaler()), # Tweedie is a GLM so sensitive to feature scale
    ("model", TweedieRegressor(
        power=1.5, # key Tweedie parameter that controls distribution type
        alpha=0.1, # regularization strength to prevent overfitting and stabilize coefficients
        link="log" # Ensures predictions are positive and match insurance severity behavior
    ))
])

| Power | Meaning |
| --- | --- |
| 1 | Poisson (counts only) |
| 2 | Gamma (positive continuous only) |
| 1-2 | Compound Poisson-Gamma (insurance data) |

Insurance typically uses **1.3–1.8**. 1.5 is PERFECT for claims

In [40]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', TweedieRegressor(alpha=0.1, link='log', power=1.5))])

In [41]:
y_pred = pipeline.predict(X_test)

In [42]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Tweedie Severity Model Performance")
print("MAE:", mae)
print("RMSE:", rmse)

Tweedie Severity Model Performance
MAE: 85445.33308156312
RMSE: 205699.84283836168


##### Conclusion

The Tweedie model significantly improved performance over the Gamma severity model, reducing prediction error by over 60%.

The dataset has many zero claims and some large claims

- Gamma only models positive claims and ignores zero-claim structure
- Tweedie models zero + positive together, captures full insurance reality and produces pure premium directly 

In [43]:
print("Actual mean:", y_test.mean())
print("Predicted mean:", y_pred.mean())

Actual mean: 46881.147756969774
Predicted mean: 47424.92474884771


In [44]:
import joblib

joblib.dump(pipeline, "tweedie_model.pkl")

['tweedie_model.pkl']

In [45]:
feature_columns = X_train.columns
joblib.dump(feature_columns, "tweedie_feature_columns.pkl")

['tweedie_feature_columns.pkl']